In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df = pd.read_csv("../data/raw/books.csv")
df.head()

,isbn13,isbn10,title,subtitle,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count
0,9780002005883,0002005883,Gilead,NaN,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0
1,9780002261982,0002261987,Spider's Web,A Novel,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0
2,9780006163831,0006163831,The One Tree,NaN,Stephen R. Donaldson,American fiction,http://books.google.com/books/content?id=OmQaw...,Volume Two of Stephen Donaldson's acclaimed se...,1982.0,3.97,479.0,172.0
3,9780006178736,0006178731,Rage of angels,NaN,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0
4,9780006280897,0006280897,The Four Loves,NaN,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0


# Columns to use:

In [4]:
df = df[
    [
        "title",
        "authors",
        "categories",
        "description",
        "average_rating",
        "ratings_count",
        "published_year"
    ]
]

df.head()

,title,authors,categories,description,average_rating,ratings_count,published_year
0,Gilead,Marilynne Robinson,Fiction,A NOVEL THAT READERS and critics have been eag...,3.85,361.0,2004.0
1,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,A new 'Christie for Christmas' -- a full-lengt...,3.83,5164.0,2000.0
2,The One Tree,Stephen R. Donaldson,American fiction,Volume Two of Stephen Donaldson's acclaimed se...,3.97,172.0,1982.0
3,Rage of angels,Sidney Sheldon,Fiction,"A memorable, mesmerizing heroine Jennifer -- b...",3.93,29532.0,1993.0
4,The Four Loves,Clive Staples Lewis,Christian life,Lewis' work on the nature of love divides love...,4.15,33684.0,2002.0


# Handling Missing Values

In [8]:
text_cols = [
    "title",
    "authors",
    "categories",
    "description"
]

for col in text_cols:
    df[col] = df[col].fillna("")

In [9]:
df["average_rating"] = df["average_rating"].fillna(
    df["average_rating"].median()
)

df["ratings_count"] = df["ratings_count"].fillna(0)

df["published_year"] = df["published_year"].fillna(0)


In [12]:
df['title'].isnull().sum()

np.int64(0)

# Text Cleaning

In [13]:
def clean_text(text):

    # convert to lowercase
    text = str(text).lower()

    # remove punctuation
    text = re.sub(r"[^\w\s]", " ", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [14]:
df["clean_title"] = df["title"].apply(clean_text)

df["clean_authors"] = df["authors"].apply(clean_text)

df["clean_categories"] = df["categories"].apply(clean_text)

df["clean_description"] = df["description"].apply(clean_text)

# Feature Engineering

In [15]:
df["popularity_score"] = np.log1p(df["ratings_count"])

# Combined Text Feature

In [16]:
df["combined_text"] = (
    df["clean_title"] + " " +
    df["clean_title"] + " " +
    df["clean_authors"] + " " +
    df["clean_categories"] + " " +
    df["clean_categories"] + " " +
    df["clean_description"]
)

In [17]:
df = df[df["combined_text"].str.strip() != ""]


# Check

In [18]:
print(df.shape)

print(df.isnull().sum())

(6810, 13)
title                0
authors              0
categories           0
description          0
average_rating       0
ratings_count        0
published_year       0
clean_title          0
clean_authors        0
clean_categories     0
clean_description    0
popularity_score     0
combined_text        0
dtype: int64


# Saving Cleaned Data

In [20]:
df.to_csv(
    "../data/processed/cleaned_books.csv",
    index=False
)